# Stock Performance Prediction — Model 2: Temporal Convolutional Network (TCN)

**Project:** Stock Performance Prediction (RNN)  
**Prerequisites:** `01_exploratory_data_analysis.ipynb`, `02_preprocessing.ipynb`  
**Loads from:** `data/processed/aapl_sequences.npz`, `scaler.json`, `pipeline_config.json`

This notebook trains and evaluates a **Temporal Convolutional Network (TCN)** classifier for next-day direction prediction on AAPL.

### Why TCN?
A TCN uses **1D dilated causal convolutions** rather than recurrent connections. Dilation factors grow exponentially (1, 2, 4, 8, 16 ...), so a small number of layers covers the full 60-day receptive field while remaining fully parallelisable. This gives three concrete advantages over GRU for this dataset:

1. **Multi-scale pattern detection**: Each dilation factor attends to a different temporal scale simultaneously — short-range momentum at dilation 1, medium-range volatility regimes at dilation 8+. EDA confirmed volatility clustering across multiple lag scales.
2. **No vanishing gradient**: Gradients flow through residual connections rather than through time steps, making training more stable.
3. **Architectural diversity for ensemble**: The TCN and BiGRU make different types of errors. Their predictions are partially decorrelated, which improves ensemble quality when combined.

### Pipeline
```
Load artifacts → Build TCN → Train with early stopping
→ Evaluate (Val + Test) → Baselines → Save model
```

| Stage | Section |
|-------|---------|
| Setup & load | 1–2 |
| Model definition | 3 |
| Training | 4 |
| Evaluation | 5 |
| Baselines | 6 |
| Save | 7 |

## 1. Setup & Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "tensorflow"]

def ensure_packages(packages):
    missing = [p for p in packages if importlib.util.find_spec(p.replace("-", "_")) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

ensure_packages(REQUIRED)

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, accuracy_score, roc_curve
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow: {tf.__version__}")
print(f"Python:     {sys.executable}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")

## 2. Load Preprocessed Artifacts

In [ ]:
def find_project_root() -> Path:
    candidates = [Path.cwd()]
    if Path.cwd().name == "notebooks":
        candidates.append(Path.cwd().parent)
    candidates.extend(Path.cwd().parents)
    for path in candidates:
        if (path / "Stock Market Dataset").exists():
            return path
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


PROJECT_ROOT  = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "data" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:  {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Models dir:    {MODELS_DIR}")

In [ ]:
# ── Load sequence arrays ──────────────────────────────────────────────────
seq_path = PROCESSED_DIR / "aapl_sequences.npz"
if not seq_path.exists():
    raise FileNotFoundError(
        f"Sequences not found at {seq_path}.\n"
        "Run 02_preprocessing.ipynb first."
    )

data = np.load(seq_path, allow_pickle=True)
X_train, y_train = data["X_train"], data["y_train"]
X_val,   y_val   = data["X_val"],   data["y_val"]
X_test,  y_test  = data["X_test"],  data["y_test"]
feature_cols     = list(data["feature_cols"])

print("Sequence shapes loaded:")
for split, X, y in [("train", X_train, y_train),
                    ("val",   X_val,   y_val),
                    ("test",  X_test,  y_test)]:
    up_pct = y.mean() * 100
    print(f"  {split:<6}  X={X.shape}  y={y.shape}  up%={up_pct:.1f}")

print(f"\nFeatures ({len(feature_cols)}): {feature_cols}")

In [ ]:
# ── Load config & scaler ──────────────────────────────────────────────────
with open(PROCESSED_DIR / "pipeline_config.json") as f:
    cfg = json.load(f)

with open(PROCESSED_DIR / "scaler.json") as f:
    scaler = json.load(f)

LOOKBACK   = cfg["lookback_days"]   # 60
N_FEATURES = len(feature_cols)      # 7

print(f"Lookback: {LOOKBACK} days | Features: {N_FEATURES}")
print(f"Train end: {cfg['train_end']}  |  Val: {cfg['val_start']} – {cfg['val_end']}  |  Test: {cfg['test_start']} – {cfg['test_end']}")

## 3. Model Definition — Temporal Convolutional Network (TCN)

### Architecture

The TCN is built from a stack of **dilated causal residual blocks**. Each block contains:
- Two `Conv1D` layers with the **same dilation factor** and causal padding (no future leakage)
- `LayerNormalization` after each convolution (more stable than BatchNorm for time series)
- `ReLU` activations
- Spatial `Dropout` for regularisation
- A **residual (skip) connection**: if input and output channels differ, a 1×1 conv aligns them

Dilation factors: `[1, 2, 4, 8, 16]` — total receptive field = 2 × kernel_size × (1+2+4+8+16) - 1 = **186 timesteps**, comfortably covering the 60-day lookback.

```
Input (60, 7)
  └─ Conv1D projection (1×1, 32 filters)          ← align channels
       └─ Dilated Residual Block (dilation=1,  32 filters)
            └─ Dilated Residual Block (dilation=2,  32 filters)
                 └─ Dilated Residual Block (dilation=4,  32 filters)
                      └─ Dilated Residual Block (dilation=8,  64 filters)
                           └─ Dilated Residual Block (dilation=16, 64 filters)
                                └─ GlobalAveragePooling1D           ← collapse time
                                     └─ Dense (32, ReLU) + Dropout
                                          └─ Dense (1, Sigmoid)  →  P(next day UP)
```

**Why GlobalAveragePooling:** Unlike the BiGRU which collapses the temporal axis via the final recurrent step, the TCN retains the full feature map across time. GlobalAveragePooling aggregates information from all 60 timesteps equally — appropriate because each dilated block already attends to multi-scale context.

In [ ]:
def tcn_residual_block(
    x: tf.Tensor,
    filters: int,
    kernel_size: int,
    dilation_rate: int,
    dropout_rate: float,
    block_id: int,
) -> tf.Tensor:
    """
    One TCN residual block:
        [Conv1D → LayerNorm → ReLU → SpatialDropout] × 2  +  skip connection.
    Causal padding ensures no future information leaks into the prediction.
    """
    residual = x

    # ── Conv 1 ────────────────────────────────────────────────────────────
    x = layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding="causal",
        use_bias=False,
        name=f"block{block_id}_conv1",
    )(x)
    x = layers.LayerNormalization(name=f"block{block_id}_ln1")(x)
    x = layers.Activation("relu", name=f"block{block_id}_relu1")(x)
    x = layers.SpatialDropout1D(dropout_rate, name=f"block{block_id}_drop1")(x)

    # ── Conv 2 ────────────────────────────────────────────────────────────
    x = layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding="causal",
        use_bias=False,
        name=f"block{block_id}_conv2",
    )(x)
    x = layers.LayerNormalization(name=f"block{block_id}_ln2")(x)
    x = layers.Activation("relu", name=f"block{block_id}_relu2")(x)
    x = layers.SpatialDropout1D(dropout_rate, name=f"block{block_id}_drop2")(x)

    # ── Residual (skip) connection ─────────────────────────────────────────
    # 1×1 conv to align channels if residual shape differs from output shape
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(
            filters=filters, kernel_size=1,
            padding="same", use_bias=False,
            name=f"block{block_id}_skip_proj",
        )(residual)

    x = layers.Add(name=f"block{block_id}_add")([x, residual])
    return x


def build_tcn(
    lookback: int,
    n_features: int,
    filters: list = None,
    kernel_size: int = 3,
    dropout_rate: float = 0.2,
    seed: int = 42,
) -> keras.Model:
    """
    Temporal Convolutional Network for binary direction classification.

    Args:
        lookback     : sequence length (timesteps)
        n_features   : number of input features per timestep
        filters      : list of (dilation_rate, n_filters) tuples
        kernel_size  : convolutional kernel width
        dropout_rate : spatial dropout rate per block
        seed         : random seed
    """
    tf.random.set_seed(seed)

    if filters is None:
        # dilation_rate, n_filters
        filters = [
            (1,  32),
            (2,  32),
            (4,  32),
            (8,  64),
            (16, 64),
        ]

    inputs = keras.Input(shape=(lookback, n_features), name="sequence_input")

    # ── Initial 1×1 projection to match first block's filter count ─────────
    x = layers.Conv1D(
        filters=filters[0][1],
        kernel_size=1,
        padding="same",
        use_bias=False,
        name="input_projection",
    )(inputs)

    # ── Stack of dilated residual blocks ───────────────────────────────────
    for block_id, (dilation_rate, n_filters) in enumerate(filters):
        x = tcn_residual_block(
            x,
            filters=n_filters,
            kernel_size=kernel_size,
            dilation_rate=dilation_rate,
            dropout_rate=dropout_rate,
            block_id=block_id,
        )

    # ── Aggregate across timesteps ─────────────────────────────────────────
    x = layers.GlobalAveragePooling1D(name="global_avg_pool")(x)

    # ── Dense head ─────────────────────────────────────────────────────────
    x = layers.Dense(
        32, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_1"
    )(x)
    x = layers.Dropout(0.2, seed=seed, name="dense_drop")(x)

    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="TCN_DirectionClassifier")
    return model


model = build_tcn(LOOKBACK, N_FEATURES, seed=SEED)
model.summary(line_length=80)

In [ ]:
# ── Compile ───────────────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.AUC(name="auc"),
    ]
)
print("Model compiled.")
print(f"Total trainable parameters: {model.count_params():,}")

## 4. Training

Identical callback strategy to the BiGRU notebook for a fair comparison:
- `EarlyStopping` on `val_auc` (patience=15, restore best weights)
- `ReduceLROnPlateau` on `val_loss` (patience=7, factor=0.5, min_lr=1e-5)
- `ModelCheckpoint` saves best by `val_auc`

The TCN typically trains **faster** than the BiGRU because convolutions are parallelisable across timesteps.

In [ ]:
CHECKPOINT_PATH = MODELS_DIR / "tcn_best.keras"

cb_early = callbacks.EarlyStopping(
    monitor="val_auc",
    patience=15,
    mode="max",
    restore_best_weights=True,
    verbose=1,
)
cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=7,
    min_lr=1e-5,
    verbose=1,
)
cb_checkpoint = callbacks.ModelCheckpoint(
    filepath=str(CHECKPOINT_PATH),
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=0,
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[cb_early, cb_reduce_lr, cb_checkpoint],
    verbose=1,
    shuffle=False,  # preserve time order within each epoch
)

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────
hist = pd.DataFrame(history.history)
epochs_ran = len(hist)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (train_col, val_col, title) in zip(axes, [
    ("loss",     "val_loss",     "Binary Cross-Entropy Loss"),
    ("accuracy", "val_accuracy", "Accuracy"),
    ("auc",      "val_auc",      "AUC-ROC"),
]):
    ax.plot(hist[train_col], label="Train", linewidth=1.8)
    ax.plot(hist[val_col],   label="Val",   linewidth=1.8, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.axvline(hist[val_col].idxmax() if "auc" in val_col else hist[val_col].idxmin(),
               color="gray", linewidth=0.8, linestyle=":")

fig.suptitle(f"TCN — Training History ({epochs_ran} epochs)", y=1.02)
plt.tight_layout()
plt.show()

best_epoch = hist["val_auc"].idxmax()
print(f"Best epoch: {best_epoch + 1}  |  val_auc={hist['val_auc'][best_epoch]:.4f}  |  val_loss={hist['val_loss'][best_epoch]:.4f}")

## 5. Evaluation

Identical evaluation protocol to the BiGRU notebook for direct comparability.
Results are reported separately for the **validation set** (2018–2019) and **test set** (2020 Q1, COVID stress window).

**Primary metrics:** F1-score, AUC-ROC  
**Secondary:** Accuracy, Precision, Recall

In [ ]:
def evaluate_split(model, X, y_true, split_name: str, threshold: float = 0.5) -> dict:
    """Full evaluation on a single split. Returns a metrics dict."""
    y_prob  = model.predict(X, verbose=0).flatten()
    y_pred  = (y_prob >= threshold).astype(int)

    acc     = accuracy_score(y_true, y_pred)
    f1      = f1_score(y_true, y_pred, zero_division=0)
    auc     = roc_auc_score(y_true, y_prob)

    print(f"\n{'='*55}")
    print(f"  {split_name.upper()} RESULTS  (n={len(y_true)}, threshold={threshold})")
    print(f"{'='*55}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1-score : {f1:.4f}")
    print(f"  AUC-ROC  : {auc:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['DOWN (0)', 'UP (1)'], zero_division=0)}")

    return {"split": split_name, "accuracy": acc, "f1": f1, "auc": auc,
            "y_prob": y_prob, "y_pred": y_pred, "y_true": y_true}


val_results  = evaluate_split(model, X_val,  y_val,  "Validation (2018–2019)")
test_results = evaluate_split(model, X_test, y_test, "Test — COVID Stress (2020 Q1)")

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, res in zip(axes, [val_results, test_results]):
    cm = confusion_matrix(res["y_true"], res["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", ax=ax,
                xticklabels=["DOWN", "UP"], yticklabels=["DOWN", "UP"])
    ax.set_title(f"{res['split']}\nF1={res['f1']:.3f}  AUC={res['auc']:.3f}")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")

fig.suptitle("TCN — Confusion Matrices", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

for res, color in zip([val_results, test_results], ["darkorange", "crimson"]):
    fpr, tpr, _ = roc_curve(res["y_true"], res["y_prob"])
    ax.plot(fpr, tpr, label=f"{res['split']} (AUC={res['auc']:.3f})",
            color=color, linewidth=2)

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("TCN — ROC Curves")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Predicted probability distributions ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, res in zip(axes, [val_results, test_results]):
    for label, color, name in [(0, "#c0392b", "DOWN"), (1, "#27ae60", "UP")]:
        mask = res["y_true"] == label
        ax.hist(res["y_prob"][mask], bins=30, alpha=0.6,
                color=color, label=f"Actual {name}")
    ax.axvline(0.5, color="black", linewidth=1, linestyle="--", label="Threshold=0.5")
    ax.set_xlabel("Predicted P(UP)")
    ax.set_ylabel("Count")
    ax.set_title(f"{res['split']}")
    ax.legend(fontsize=8)

fig.suptitle("TCN — Predicted Probability Distributions by True Class", y=1.02)
plt.tight_layout()
plt.show()

## 6. Baseline Comparisons

Same baselines as the BiGRU notebook — ensures direct comparability.

| Baseline | Description |
|----------|-------------|
| Always UP | Predicts 1 (up) for every sample |
| Always DOWN | Predicts 0 (down) for every sample |
| Last direction | Predicts the direction of the most recent log_return in the lookback window |

In [ ]:
def baseline_metrics(y_true, y_pred, name: str) -> dict:
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 4),
    }


rows = []
for split_name, X, y in [("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    last_return = X[:, -1, 0]  # most recent day's log_return (feature index 0)
    last_dir    = (last_return > 0).astype(int)

    tcn_pred = (model.predict(X, verbose=0).flatten() >= 0.5).astype(int)

    for row in [
        baseline_metrics(y, np.ones_like(y, dtype=int),  f"[{split_name}] Always UP"),
        baseline_metrics(y, np.zeros_like(y, dtype=int), f"[{split_name}] Always DOWN"),
        baseline_metrics(y, last_dir,                    f"[{split_name}] Last Direction"),
        {"Model": f"[{split_name}] TCN (ours)",
         "Accuracy": round(accuracy_score(y, tcn_pred), 4),
         "F1":       round(f1_score(y, tcn_pred, zero_division=0), 4)},
    ]:
        rows.append(row)

baseline_df = pd.DataFrame(rows)
print("\nBaseline comparison:")
display(baseline_df.set_index("Model"))

In [ ]:
# ── Bar chart: F1 comparison ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
colors_bar = [
    "#e67e22" if "TCN" in m else "#bdc3c7"
    for m in baseline_df["Model"]
]
ax.barh(baseline_df["Model"], baseline_df["F1"], color=colors_bar, edgecolor="white")
ax.axvline(0.5, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("F1-Score")
ax.set_title("TCN vs Baselines — F1-Score")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Save Model & Results

In [ ]:
# ── Save final model ──────────────────────────────────────────────────────
final_path = MODELS_DIR / "tcn_final.keras"
model.save(final_path)
print(f"Model saved: {final_path}")

# ── Save predictions for ensemble notebook ────────────────────────────────
preds = {
    "val_prob":  val_results["y_prob"].tolist(),
    "val_pred":  val_results["y_pred"].tolist(),
    "val_true":  val_results["y_true"].tolist(),
    "test_prob": test_results["y_prob"].tolist(),
    "test_pred": test_results["y_pred"].tolist(),
    "test_true": test_results["y_true"].tolist(),
    "metrics": {
        "val":  {"accuracy": val_results["accuracy"],  "f1": val_results["f1"],  "auc": val_results["auc"]},
        "test": {"accuracy": test_results["accuracy"], "f1": test_results["f1"], "auc": test_results["auc"]},
    }
}
preds_path = MODELS_DIR / "tcn_predictions.json"
with open(preds_path, "w") as f:
    json.dump(preds, f, indent=2)
print(f"Predictions saved: {preds_path}")

# ── Summary table ─────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {"Split": "Validation (2018–2019)",       **preds["metrics"]["val"]},
    {"Split": "Test — COVID Stress (2020 Q1)", **preds["metrics"]["test"]},
])
print("\n── TCN Final Results ──")
display(summary.set_index("Split").round(4))

---

## Summary

This notebook trained and evaluated a **Temporal Convolutional Network** with 5 dilated residual blocks (dilation rates: 1, 2, 4, 8, 16; filters: 32, 32, 32, 64, 64) on AAPL next-day direction prediction using 60-day sequences of 7 engineered features. The total receptive field spans 186 timesteps, well beyond the 60-day lookback window.

**Key artifacts produced:**
- `data/models/tcn_best.keras` — best checkpoint (by val AUC)
- `data/models/tcn_final.keras` — final model
- `data/models/tcn_predictions.json` — probabilities and metrics for ensemble use

**Both models are now trained.** The `bigru_predictions.json` and `tcn_predictions.json` files contain the raw probability outputs needed to combine the two into an ensemble.